<a href="https://colab.research.google.com/github/hindxb/FDS/blob/main/Notebooks/Project/FDS_KNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

by fatima

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
uploaded= files.upload()



Saving chicago_traffic_imputed(in).csv to chicago_traffic_imputed(in).csv


In [ ]:
mydata=pd.read_csv('chicago_traffic_imputed(in).csv',low_memory=False)
mydata.head()
mydata.shape

(1018752, 25)

In [ ]:
mydata = mydata.sort_values(by=['SEGMENT_ID','TIME']).copy()
features=['HOUR',
    'DAY_OF_WEEK',
    'MONTH',
    'SEGMENT_ID',
    'LENGTH',
    'BUS_COUNT',
    'MESSAGE_COUNT',
    'START_LATITUDE',
    'START_LONGITUDE',
    'END_LATITUDE',
    'END_LONGITUDE']
# future targets
mydata['target_15']=mydata.groupby('SEGMENT_ID')['SPEED_FILLED'].shift(-1)
mydata['target_30']=mydata.groupby('SEGMENT_ID')['SPEED_FILLED'].shift(-2)
mydata['target_45']=mydata.groupby('SEGMENT_ID')['SPEED_FILLED'].shift(-3)
mydata['target_60']=mydata.groupby('SEGMENT_ID')['SPEED_FILLED'].shift(-4)
# remove rows without future target values
mydata=mydata.dropna(subset=['target_15','target_30','target_45','target_60'])
inputs_x=mydata[features]


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

targets={'15 minutes':'target_15','30 minutes':'target_30','45 minutes':'target_45','60 minutes':'target_60'}
results=[]
for horizon, target in targets.items():
  output_y=mydata[target]
  X_train, X_test, y_train, y_test = train_test_split(inputs_x,output_y,test_size=0.2,random_state=100)
  scaler=StandardScaler()
  input_x_train=scaler.fit_transform(X_train)
  input_x_test=scaler.transform(X_test)
  knn=KNeighborsRegressor(n_neighbors=5)
  knn.fit(input_x_train,y_train)
  y_pred=knn.predict(input_x_test)

  mae=mean_absolute_error(y_test,y_pred)
  mse=mean_squared_error(y_test,y_pred)
  rmse = np.sqrt(mse)
  r2=r2_score(y_test,y_pred)
  results.append([horizon,mae,rmse,r2])

results_df=pd.DataFrame(results,columns=['horizon','mae','rmse','r2'])
print(results_df)


      horizon       mae      rmse        r2
0  15 minutes  3.127639  4.215009  0.272231
1  30 minutes  3.160640  4.267617  0.253087
2  45 minutes  3.167085  4.268942  0.252130
3  60 minutes  3.170252  4.279419  0.248275
